If LLM is not availabe for a specific language or very specific task, we can train it from scratch<br /> But firsly we need to train a new tokenzer.<br /> To train tokenizer we not use SGD, we statical process which are deterministic.

We train tokenizers for a Python progamming language.

In [ ]:
from datasets import load_dataset

In [ ]:
# This can take a few minutes to load, so grab a coffee or tea while you wait!
raw_datasets = load_dataset("code_search_net", "python")

In [ ]:
raw_datasets["train"]

In [ ]:
print(raw_datasets["train"][123456]["whole_func_string"])

We create generator to not load everything in memory.

In [ ]:
training_corpus = (
    raw_datasets["train"][i : i + 1000]["whole_func_string"]
    for i in range(0, len(raw_datasets["train"]), 1000)
)

In [ ]:
training_corpus

In [ ]:
# When we execute looping through generator second time it will be empty, because generator is exhausted after first loop.
gen = (i for i in range(10))
print(list(gen))
print(list(gen))

In [ ]:
# We create generator function, now each time we call we will get new generator.
def get_training_corpus():
    return (
        raw_datasets["train"][i : i + 1000]["whole_func_string"]
        for i in range(0, len(raw_datasets["train"]), 1000)
    )


training_corpus = get_training_corpus()

In [ ]:
# Or with yield (for more complex processing)
def get_training_corpus():
    dataset = raw_datasets["train"]
    for start_idx in range(0, len(dataset), 1000):
        samples = dataset[start_idx : start_idx + 1000]
        yield samples["whole_func_string"]

## Training a new tokenizer

To train new tokenizer from old one for our needs, we first need to load old tokenizer. <br />
In this case we will use `gpt2` tokenizer.

In [ ]:
from transformers import AutoTokenizer

old_tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [ ]:
old_tokenizer

In this case the only thing that we are changing is a vocubulary.

In [ ]:
example = '''def add_numbers(a, b):
    """Add the two numbers `a` and `b`."""
    return a + b'''

tokens = old_tokenizer.tokenize(example)
tokens

As we can see our tokenizer not parse code efficienltly and in some cases it's weirdly split code into tokens. <br />
To fix this we need to train new tokenizer with new vocabulary. <br />

In [ ]:
tokenizer = old_tokenizer.train_new_from_iterator(training_corpus, 52000)

In [ ]:
tokenizer

In [ ]:
example = '''def add_numbers(a, b):
    """Add the two numbers `a` and `b`."""
    return a + b'''

tokens = tokenizer.tokenize(example)
tokens

In [ ]:
print(len(tokens))
print(len(old_tokenizer.tokenize(example)))

In [ ]:
example = """class LinearLayer():
    def __init__(self, input_size, output_size):
        self.weight = torch.randn(input_size, output_size)
        self.bias = torch.zeros(output_size)

    def __call__(self, x):
        return x @ self.weights + self.bias
    """
tokenizer.tokenize(example)

When we trained tokenizer on our vocobulary we see that it learns specifics about vocubalary and it now splits code into tokens in more efficient way. <br />


Now we can save our tokenizer.

In [ ]:
tokenizer.save_pretrained("code-search-net-tokenizer")

Uploading our tokenizer to HuggingFace Hub

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
tokenizer.push_to_hub("code-search-net-tokenizer")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("osrokas/code-search-net-tokenizer")


Tokenizer types in HF:
 - Slow tokenizer - written in Python, easy to understand and modify, but slow.
 - Fast tokenizer - written in Rust, much faster, but harder to understand and modify. (keep track the original span of text - offset mapping)
  
We can see a difference between speed of tokenization when we have a lot of data. <br />
Slow tokenizer works faster with small amount of data.

Outut of tokenizer is BatchEncoding object.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
berta_encoding = tokenizer(example)
print(type(berta_encoding))

In [ ]:
# Check if the tokenizer is a fast tokenizer
print(tokenizer.is_fast)
print(berta_encoding.is_fast)

In [ ]:
# Check tokens
berta_encoding.tokens()

In [ ]:
# Check word ids for each token (which word it belongs to)
berta_encoding.word_ids()

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
encoding = tokenizer(example)
print(type(encoding))

In [ ]:
encoding.tokens()

Different tokenizer split words differenly.

In [ ]:
# Check word ids for each token (which word it belongs to)
print(berta_encoding.word_ids())
print(encoding.word_ids())

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
example = "My name is Sylvain and I work at Hugging Face in Brooklyn. We really love working in Brooklyn."
encoding = tokenizer(example)
print(type(encoding))

In [ ]:
encoding.sequence_ids()

Mapping tokens to words.

In [ ]:
start, end = encoding.word_to_chars(3)
example[start:end]

## Named Entity Recognition (NER)

Detects which parts of text are entities like people, places, organizations, etc. <br />

In [ ]:
from transformers import pipeline

token_classifier = pipeline("token-classification")
token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")

Group tokens into words with NER

We need to define aggregation strategy to group tokens into words. <br />
 - first - take the first token of the word
 - simple - average of whole entity 
 - average - average of words (not whole entity)
 - max - take the max of all tokens of the word

In [ ]:
from transformers import pipeline

token_classifier = pipeline("token-classification", aggregation_strategy="simple")
token_classifier("My name is Sylvain and I work at Hugging Face in Brooklyn.")

Now we apply this with tokenizer without pipeline

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_checkpoint = "dbmdz/bert-large-cased-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint)

example = "My name is Sylvain and I work at Hugging Face in Brooklyn."
inputs = tokenizer(example, return_tensors="pt")
outputs = model(**inputs)

In [ ]:
print(inputs["input_ids"].shape)
print(outputs.logits.shape)

In [ ]:
print(outputs.logits)

To get classes of entitities we need to convert logits to probabilities applying softmax. <br />
Then we can apply argmax to get the most probable class for a token. <br />

In [ ]:
# Get all probablities for each class of each token
import torch

probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].tolist()
probabilities

In [ ]:
# Get the most probable class for each token
predictions = outputs.logits.argmax(dim=-1)[0].tolist()
print(predictions)

- O - outside of entity
- B - beginning of entity
- I - inside of entity

In [ ]:
model.config.id2label

In [ ]:
tokens = inputs.tokens()
tokens

In [ ]:
results = []

# Iterate through predictions
for idx, pred in enumerate(predictions):
    # Get label of the predicted class
    label = model.config.id2label[pred]
    # If the label is not "O" (outside of entity) we save it to results
    if label != "O":
        # Append entity label, probability of the predicted class and the word to results
        results.append(
            {"entity": label, "score": probabilities[idx][pred], "word": tokens[idx]}
        )



In [ ]:
print(results)

## Grouping tokens into entities

In [ ]:
example[33:45]

In [ ]:
import numpy as np

In [ ]:
# Tokenize the example and get offsets
inputs_with_offsets = tokenizer(example, return_offsets_mapping=True)
inputs_with_offsets

In [ ]:
# Get tokens from embeddings
tokens = inputs_with_offsets.tokens()
tokens

In [ ]:
# Get offsets for each token
offsets = inputs_with_offsets["offset_mapping"]
offsets

Calculate average score for each entity and group tokens into entities

In [ ]:
results = []
# Define an index to keep track of our position in the predictions list
idx = 0
# iterate through predictions and group tokens into entities
while idx < len(predictions):
    # Predicted class for the current token
    pred = predictions[idx]
    # Get label of the predicted class
    label = model.config.id2label[pred]
    # If the label is not "O" (outside of entity) we save it to results
    if label != "O":
        # Remove the B- or I-
        label = label[2:]
        # Get offset 
        start, _ = offsets[idx]
        # Grab all the tokens labeled with I-label
        all_scores = []
        while (
            idx < len(predictions)
            and model.config.id2label[predictions[idx]] == f"I-{label}"
        ):
            all_scores.append(probabilities[idx][pred])
            _, end = offsets[idx]
            idx += 1

        # The score is the mean of all the scores of the tokens in that grouped entity
        score = np.mean(all_scores).item()
        word = example[start:end]
        results.append(
            {
                "entity_group": label,
                "score": score,
                "word": word,
                "start": start,
                "end": end,
            }
        )
    idx += 1

print(results)

## Tokenizer in QA pipelines

In [ ]:
from transformers import pipeline

question_answerer = pipeline("question-answering")
context = """
🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch, and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""
question = "Which deep learning libraries back 🤗 Transformers?"
question_answerer(question=question, context=context)

This pipeline is also sutable for a long context question answering. <br />

In [ ]:
long_context = """
🤗 Transformers: State of the Art NLP

🤗 Transformers provides thousands of pretrained models to perform tasks on texts such as classification, information extraction,
question answering, summarization, translation, text generation and more in over 100 languages.
Its aim is to make cutting-edge NLP easier to use for everyone.

🤗 Transformers provides APIs to quickly download and use those pretrained models on a given text, fine-tune them on your own datasets and
then share them with the community on our model hub. At the same time, each python module defining an architecture is fully standalone and
can be modified to enable quick research experiments.

Why should I use transformers?

1. Easy-to-use state-of-the-art models:
  - High performance on NLU and NLG tasks.
  - Low barrier to entry for educators and practitioners.
  - Few user-facing abstractions with just three classes to learn.
  - A unified API for using all our pretrained models.
  - Lower compute costs, smaller carbon footprint:

2. Researchers can share trained models instead of always retraining.
  - Practitioners can reduce compute time and production costs.
  - Dozens of architectures with over 10,000 pretrained models, some in more than 100 languages.

3. Choose the right framework for every part of a model's lifetime:
  - Train state-of-the-art models in 3 lines of code.
  - Move a single model between TF2.0/PyTorch frameworks at will.
  - Seamlessly pick the right framework for training, evaluation and production.

4. Easily customize a model or an example to your needs:
  - We provide examples for each architecture to reproduce the results published by its original authors.
  - Model internals are exposed as consistently as possible.
  - Model files can be used independently of the library for quick experiments.

🤗 Transformers is backed by the three most popular deep learning libraries — Jax, PyTorch and TensorFlow — with a seamless integration
between them. It's straightforward to train your models with one before loading them for inference with the other.
"""
question_answerer(question=question, context=long_context)

## How it works

Tokenize inputs and pass them through QA model

In [ ]:
# Load tokenizer and model for question answering
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_checkpoint = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForQuestionAnswering.from_pretrained(model_checkpoint)

In [ ]:
# Tokenization of inputs
inputs = tokenizer(question, context, return_tensors="pt")
inputs

Model outputs part of the text that is consider to be an answer to the question. <br />

In [ ]:
# Pass tokens to model
outputs = model(**inputs)

# Printing logits
outputs

Getting answer from start to end from the logits

In [ ]:
start_logits = outputs.start_logits
end_logits = outputs.end_logits
print(start_logits.shape, end_logits.shape)

In [ ]:
start_logits

Before applying softmax we need to remove question tokens from the logits.

In [ ]:
import torch

sequence_ids = inputs.sequence_ids()
sequence_ids

In [ ]:
# Mask everything apart from the tokens of the context
mask = [i != 1 for i in sequence_ids]
mask

In [ ]:
# Unmask the [CLS] token
mask[0] = False

In [ ]:
mask = torch.tensor(mask)[None]

In [ ]:
mask

In [ ]:
start_logits[mask] = -10000
end_logits[mask] = -10000

In [ ]:
start_logits

Great, we just masked tokens that we don't want to predict. It's the text that is predicted as the answer to the question.

In [ ]:
start_probabilities = torch.nn.functional.softmax(start_logits, dim=-1)[0]
end_probabilities = torch.nn.functional.softmax(end_logits, dim=-1)[0]

In [ ]:
start_probabilities

In [ ]:
# Compute all posibilities of start and end tokens
scores = start_probabilities[:, None] * end_probabilities[None, :]


In [ ]:
# Apply masking
scores = torch.triu(scores)

In [ ]:
scores

From start to end locations we calculate predictions of the answer.

In [ ]:
# Calcuate the most probable start and end token combination
max_index = scores.argmax().item()
start_index = max_index // scores.shape[1]
end_index = max_index % scores.shape[1]
print(scores[start_index, end_index])

In [ ]:
inputs_with_offsets = tokenizer(question, context, return_offsets_mapping=True)
offsets = inputs_with_offsets["offset_mapping"]

In [ ]:
inputs_with_offsets

In [ ]:
context

In [ ]:
offsets

In [ ]:
start_char, _ = offsets[start_index]
_, end_char = offsets[end_index]
answer = context[start_char:end_char]

In [ ]:
answer

Now we can create final answer dictionary with answer of the question, start and end postions from wherw we consider as an answer to the question from the context, and finally the score of the answer.

In [ ]:
result = {
    "answer": answer,
    "start": start_char,
    "end": end_char,
    "score": scores[start_index, end_index],
}
print(result)

## Handling long contexts
question_answerer max token length is 384, but our input is 461 tokens. <br />
It means that context will be truncated and we will lose some information. <br />


In [ ]:
inputs = tokenizer(question, long_context)
print(len(inputs["input_ids"]))

In [ ]:
inputs = tokenizer(question, long_context, max_length=384, truncation="only_second")
print(tokenizer.decode(inputs["input_ids"]))

To do that we need to split context into smaller chunks and provide some overlaping between them

In [ ]:
sentence = "This sentence is not too long but we are going to split it anyway."
inputs = tokenizer(
    sentence, truncation=True, return_overflowing_tokens=True, max_length=6, stride=2
)

for ids in inputs["input_ids"]:
    print(tokenizer.decode(ids))

In [ ]:
print(inputs.keys())

In [ ]:
print(inputs["overflow_to_sample_mapping"])

It's more useful when we have couple sentences and we need to split them into smaller chunks without loosing information. <br />
We also aware that our input is always same size (torch tensors requires that all inputs have same size).

In [ ]:
sentences = [
    "This sentence is not too long but we are going to split it anyway.",
    "This sentence is shorter but will still get split.",
]
inputs = tokenizer(
    sentences, truncation=True, return_overflowing_tokens=True, max_length=6, stride=2
)

print(inputs["overflow_to_sample_mapping"])

Now we can pass out long context through our tokenizer with strategies mentioned above

In [ ]:
inputs = tokenizer(
    question,
    long_context,
    stride=128,
    max_length=384,
    padding="longest",
    truncation="only_second",
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
)

In [ ]:
_ = inputs.pop("overflow_to_sample_mapping")
offsets = inputs.pop("offset_mapping")

inputs = inputs.convert_to_tensors("pt")
print(inputs["input_ids"].shape)

In [ ]:
inputs['input_ids'][1]

Above we have 2 chunks with 384 tokens size, that overlaps each other by 128 tokens. <br />

In [ ]:
outputs = model(**inputs)

start_logits = outputs.start_logits
end_logits = outputs.end_logits
print(start_logits.shape, end_logits.shape)

Because we split context into 2 chunks, our outpus is also 2 chunks of logits. <br />
Now we should mask all tokens that are not the answers and also mask tokens that are created for padding. <br />

In [ ]:
sequence_ids = inputs.sequence_ids()
# Mask everything apart from the tokens of the context
mask = [i != 1 for i in sequence_ids]
# Unmask the [CLS] token
mask[0] = False
# Mask all the [PAD] tokens
mask = torch.logical_or(torch.tensor(mask)[None], (inputs["attention_mask"] == 0))

start_logits[mask] = -10000
end_logits[mask] = -10000

In [ ]:
start_logits[1]

And calculate probabilities for each chunks using softmax

In [ ]:
start_probabilities = torch.nn.functional.softmax(start_logits, dim=-1)
end_probabilities = torch.nn.functional.softmax(end_logits, dim=-1)

In [ ]:
start_probabilities[1]

Now we are calculating the score of the answer for each chunk and store it in candidates variable

In [ ]:
candidates = []
for start_probs, end_probs in zip(start_probabilities, end_probabilities):
    scores = start_probs[:, None] * end_probs[None, :]
    idx = torch.triu(scores).argmax().item()

    start_idx = idx // scores.shape[1]
    end_idx = idx % scores.shape[1]
    score = scores[start_idx, end_idx].item()
    candidates.append((start_idx, end_idx, score))

print(candidates)

Finally we get dictionary as before but with two candidates

In [ ]:
for candidate, offset in zip(candidates, offsets):
    start_token, end_token, score = candidate
    start_char, _ = offset[start_token]
    _, end_char = offset[end_token]
    answer = long_context[start_char:end_char]
    result = {"answer": answer, "start": start_char, "end": end_char, "score": score}
    print(result)

As we can see second candidate has higher score, so we can consider it as an answer to the question.